# Function define

In [8]:
!pip install quandl
!pip install PyPortfolioOpt==1.5.5
!pip install arch
!pip install numpy

!pip install scikit-learn==1.2.2
import sklearn
sklearn.__version__

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from datetime import datetime as dt
import datetime
import quandl
import math
import pypfopt
import random
from statsmodels.stats.diagnostic import het_arch
from arch import arch_model
from pypfopt import EfficientFrontier,objective_functions
from sklearn.decomposition import PCA
from sklearn.mixture import GaussianMixture
from sklearn.cluster import KMeans
from sklearn import metrics
from copy import deepcopy
from sklearn.metrics import davies_bouldin_score
from collections import Counter

pypfopt.__version__

!pip install yfinance==0.2.40
from pandas_datareader import data as pdr
import yfinance as yf
yf.pdr_override()

def modify_date(date):
    return date[:10]

def data_cleaning(data):
    if (data.isna()).any().any():
        missing = np.where(data.isna())
        rows=missing[0]
        cols=missing[1]
        print(rows)
        for row in range(len(rows)):
            col=cols[row]
            if row == len(data)-1:
                data.iloc[row,col]=data.iloc[row-1,col]**2/data.iloc[row-2,col]
            elif row ==0:
                data.iloc[row,col]=data.iloc[row+1,col]**2/data.iloc[row+2,col]
            else:
                data.iloc[row,col]=math.sqrt(data.iloc[row-1,col]*data.iloc[row+1,col])
        print(data)

def get_price_years_ago(data, date,time):
    year_ago=date-datetime.timedelta(days=int(time*365))
    price_year_ago=data[data.index<=year_ago]
    return price_year_ago

def max_drawdown_absolute(returns):
    r = returns.add(1).cumprod()
    dd = r.div(r.cummax()).sub(1)
    mdd = dd.min()
    end = dd.argmin()
    start = r.iloc[:end].argmax()
    return abs(mdd), start, end

def fit_garch(data,p,q):
    lag = max(p,q)
    length =res.shape[0]
    am = arch_model(res, vol='Garch', p=1, o=0, q=1, dist='Normal')
    split_date = one_year_monthly_return.index.values[lag-1]
    results = am.fit(last_obs=split_date)
    forecasts = results.forecast(horizon=1, start = length-lag, method='simulation', simulations=1000)
    sims = forecasts.variance
    std = np.sqrt(np.sum(sims)*12/lag)
    return std

def post_allocation(dfMVOweight):

    dfMVOweight['equal']=1/len(dfMVOweight)
    numofstocks=len(dfMVOweight)
    allocationweight=allocation.loc[numofstocks,:]
    dfMVOweight['proportion']=''
    dfMVOweight['adjusted proportion']=''
    dfMVOweight['direct']=''
    residual=0
    for i in range(numofstocks):
        dfMVOweight.iloc[i,4]=dfMVOweight.iloc[i,1]/np.sum(dfMVOweight['PR_SR'])
        if (dfMVOweight.iloc[i,4]+residual/(numofstocks-i))>allocationweight[0]:
            dfMVOweight.iloc[i,5]=allocationweight[0]
        else:
            dfMVOweight.iloc[i,5]=(dfMVOweight.iloc[i,4]+residual/(numofstocks-i))
        residual+=dfMVOweight.iloc[i,4]-dfMVOweight.iloc[i,5]
    for i in range(int(allocationweight[1])):
        dfMVOweight.iloc[i,6]=allocationweight[0]
    for i in range(int(allocationweight[1]),int(allocationweight[1]+allocationweight[3])):
        dfMVOweight.iloc[i,6]=allocationweight[2]
    return dfMVOweight

def output(weight):
    if stock_type == "S&P100":
      OEX=pdr.get_data_yahoo("^OEX", start=start_swan_date, end=end_swan_date)["Adj Close"]

    elif stock_type == "S&P500":
      OEX=pdr.get_data_yahoo("SPY", start=start_swan_date, end=end_swan_date)["Adj Close"]

    stock_price=pdr.get_data_yahoo(list(weight.index), start=start_swan_date, end=end_swan_date)["Adj Close"]
    OEX.to_csv("oex.csv")
    injected_result = pd.DataFrame(index = stock_price.index )
    S1=np.array(10000*weight.T)/np.array(stock_price.iloc[0,:]).T
    portfolio_value = stock_price@S1.T
    injected_result["portfolio_value"]=portfolio_value
    S2 = 10000/float(OEX[0])
    injected_result["oex"]=S2*OEX
    return injected_result

In [9]:
def get_portfolio_PR(result,n_preselected_stock=40,n_seed=14,max_weight=0.14):
  # Preselection parameters set up
  #method = "K_means_multiple"# or avg or anyother method
  method = "K_means_multiple"
  stock_type = "S&P100"
  end_swan = "Swan9"         #equals the last swan before the swan we are going to predict
  start_swan="Swan5"
  start = dt(2020,11,1)
  end = dt(2022,1,3)
  selected_stock_number = n_preselected_stock     #number of stock preselected
  start_swan_date = dt(2022,1,3)
  end_swan_date = dt(2023,1,1)
  w1 = max_weight
  w2 = max_weight
  MA_days = 30
  N = 7 # This parameter is set to help classification method like K-means
  Min_appear = 2
  Seeds = n_seed

  #read all stocks and their swan beta
  if stock_type == "S&P100":
      portfolio_beta = pd.read_excel("All_SwanBeta100.xlsx",index_col=0)
      #portfolio_beta = pd.read_excel("BetaMatrix S&P100 Cathy(Yijue) 2022.xlsx",index_col=0)
  else:
      portfolio_beta = pd.read_excel("All_SwanBeta500.xlsx",index_col=0)

  optdata = pd.read_excel("optdata_SP100_20220103_a.xlsx",index_col=0)
  portfolio_beta = portfolio_beta[portfolio_beta.index.isin(optdata.index)]

  selected_swans_var=portfolio_beta.loc[:,start_swan:end_swan]
  portfolio_beta =portfolio_beta[ selected_swans_var.T.isnull().sum()<selected_swans_var.shape[1]-1]

  corr=portfolio_beta.corr()
  selected_corr=corr.loc["Swan1":end_swan,"Swan1":end_swan]
  corr1={}
  for i in range(selected_corr.shape[1]):
      for j in range(i+1,selected_corr.shape[0]):
          selected_corr.iloc[j,i]=np.nan

  selected_corr

  avg=selected_corr.mean(axis=1)
  weight_swan = avg/sum(avg)
  selected_beata=portfolio_beta.loc[:,start_swan:end_swan].fillna(0)
  #portfolio_beta["corr_beta"]=selected_beata@weight_swan
  #portfolio_beta=portfolio_beta.sort_values(by=["corr_beta"])

  portfolio_beta

  preselected_stock = portfolio_beta.loc[:,:]
  preselected_stock = preselected_stock.sort_values(by=['Swan10D'])
  preselected_stock = preselected_stock.iloc[:selected_stock_number,:]
  preselected_stock['avg'] = preselected_stock['Swan10D']

  #using average step
  '''
  preselected_stock

  preselected_stock['avg'] = (preselected_stock['Swan4'] + preselected_stock['Swan5']+preselected_stock['Swan6']+preselected_stock['Swan7']+preselected_stock['Swan8'])/5
  '''

  preselected_stock["PR"]=1-preselected_stock["avg"]
  preselected_stock.to_csv('preselected_stocks.csv')
  preselected_stock

  """# Step 2 - Download data and adjuste price based on Swan Beta"""

  #next line is used to input preselected stock from excel, if you are going to preselect by your self, use this blocks and note next one.
  #selected_stock_list =  pd.read_excel("19_stickers.xlsx",header = None).values[:,0]
  #swan_beta=pd.read_excel("19_stickers.xlsx",header = None,index_col=0).iloc[:,0]

  #Set the paramter to download tradeinfo from Yahoo Finance
  selected_stock=preselected_stock
  selected_stock_list=selected_stock.index.values
  swan_beta=preselected_stock.loc[selected_stock_list,"avg"]
  print(selected_stock_list)



  #download data from yahoo finance
  yahooDataFrame= pdr.get_data_yahoo(list(selected_stock_list), start=start, end=end)["Adj Close"]

  yahooDataFrame.to_csv('yahooDataFrame.csv')
  yahooDataFrame=yahooDataFrame[selected_stock_list]
  yahooDataFrame

  #set parameters to calculate predict market drop
  market_drop_info = pd.read_excel("MarketDD.xlsx",index_col=0)
  #swan_to_predict=100
  total_day=yahooDataFrame.shape[0]
  t1=int(round(market_drop_info.loc[:end_swan,"T1"].mean(),0))
  t1=int((np.array(market_drop_info.loc[:end_swan,"T1"])*np.array(weight_swan)).sum())
  #t2=total_day-t1
  t2=int(round(market_drop_info.loc[:end_swan,"T2"].mean(),0))
  dd=market_drop_info.loc[:end_swan,"Actual DD"].mean()
  t0=int(total_day-t1-t2)

  #calculate predict market drop
  md = [] #Make market drop data list named md


  j = 0 #Start from 0
  inc = dd/(t1-1) #Calculate unit increasing market drop
  for i in range(t1): #Append market drop data for t1
      md.append(round(j,4))
      j = j + inc

  base = (t2+1)**(1/(dd*100)) #Calculate log base
  l1 = [] #Make decreasing market drop list
  for i in range(t2):
      l1.append(math.log((i+2),base)/100)

  for i in range(t2): #Append market drop data for t2
      md.append(round((dd-l1[i]),4))

  for i in range(t0):
      md.append(0)

  market_drop = pd.DataFrame(md) #Change list into dataframe
  #market_drop = market_drop.rename(columns={0: 'Swan Drop spy'}) #Change column name
  market_drop1 = pd.read_csv("trainingMarketdrop.csv")

  market_drop.to_csv("market_drop.csv")

  market_drop

  swan_drop_pct=np.array(swan_beta)*np.array(market_drop)
  yahooDataFrame=pd.DataFrame((1-swan_drop_pct)*np.array(yahooDataFrame),index=yahooDataFrame.index,columns=yahooDataFrame.columns)

  yahooDataFrame=yahooDataFrame.dropna(how="all",axis = 1)
  selected_stock_list=yahooDataFrame.columns.values
  selected_stock_number = yahooDataFrame.shape[1]
  selected_stock_list

  yahooDataFrame.to_csv('yahooDataFrame.csv')
  yahooDataFrame

  """# Step3 - MVO"""

  #Set the paramter for MVO process
  Portfolio_Date=end

  #data cleaning - filling missing data in yahooDataFrame
  data_cleaning(yahooDataFrame)

  #calculated rf
  #bond_yield = pd.read_csv("bond_yield.csv", index_col=0)
  #bond_yield=np.log(bond_yield/100+1)
  #date=list(map(str,bond_yield.index.values))
  #date=list(map(modify_date,date))
  #week=pd.to_datetime(date).strftime("%W").values
  #week_1=np.roll(week,1)
  #week_difference=np.where(week!=week_1)
  #week_difference=np.delete(week_difference,0,None)
  #rf_week=bond_yield.iloc[week_difference-1]

  # Calculated R for 1 year and 0.5 year
  today_price = yahooDataFrame.iloc[-MA_days:,:]
  one_year_ago_data = get_price_years_ago(yahooDataFrame,end,1)
  one_year_ago_date = one_year_ago_data.index.values[-1]
  half_year_ago_data = get_price_years_ago(yahooDataFrame,end,0.5)
  half_year_ago_date = half_year_ago_data.index.values[-1]
  one_year_ago_price=yahooDataFrame.loc[one_year_ago_date:,:]
  one_year_ago_price=yahooDataFrame.iloc[:MA_days,:]
  half_year_ago_price=yahooDataFrame.loc[half_year_ago_date:,:]
  half_year_ago_price=yahooDataFrame.iloc[:MA_days,:]

  R_mv_one_year=np.mean(pd.DataFrame(np.array(today_price)/np.array(one_year_ago_price)-1,index=today_price.index,columns=today_price.columns),axis=0)
  R_mv_half_year=np.mean(pd.DataFrame(np.array(today_price)/np.array(half_year_ago_price)-1,index=today_price.index,columns=today_price.columns),axis=0)

  # Calculate MA
  MA_Data=yahooDataFrame.rolling(MA_days).mean().dropna(axis=0)
  MA_Data=MA_Data.loc[one_year_ago_date:,:]
  MA_now = MA_Data.iloc[-1,:]
  MA_one_year_ago = MA_Data.iloc[0,:]
  MA_half_year_ago = MA_Data.loc[half_year_ago_date,:]

  # Calculate daily return
  daily_return = yahooDataFrame.pct_change().dropna()
  log_daily_return = np.log(1+daily_return)

  # Get week end stock price data
  stock_date = yahooDataFrame.index.values
  week=pd.to_datetime(stock_date).strftime("%W").values
  week_1=np.roll(week,1)
  week_difference=np.where(week!=week_1)
  week_difference=np.delete(week_difference,0,None)
  # Calculate weekly return

  weekly_return=yahooDataFrame.iloc[week_difference-1].pct_change().dropna()
  log_weekly_return = np.log(1+weekly_return)

  # Get month end stock price data
  stock_date = yahooDataFrame.index.values
  month=pd.to_datetime(stock_date).strftime("%m").values
  month_1=np.roll(month,1)
  month_difference=np.where(month!=month_1)
  month_difference=np.delete(month_difference,0,None)
  # Calculate monthly return
  monthly_return=yahooDataFrame.iloc[month_difference-1].pct_change().dropna()
  log_monthly_return = np.log(1+monthly_return)

  # Get semi-month end stock price data
  semi_month = []
  for i in range(1,len(month_difference)):
      semi_month.append((month_difference[i]-month_difference[i-1])//2+month_difference[i-1])
      semi_month.append(month_difference[i]-1)
  # Calculate semi-monthly return
  semi_monthly_return=yahooDataFrame.iloc[semi_month].pct_change().dropna()
  log_semi_monthly_return = np.log(1+semi_monthly_return)

  # Calculated Max Dropdown for each stock
  stock_return_one_year=daily_return.loc[one_year_ago_date:,:]
  mdd=[]
  for column in stock_return_one_year.columns:
      mdd .append(max_drawdown_absolute(stock_return_one_year.loc[:,column]))
  mdd=pd.DataFrame(mdd,index = stock_return_one_year.columns,columns=["MAXDD","start_index","end_index"])
  mar=R_mv_one_year/mdd.iloc[:,0]

  #calculate weekly sd for each stock
  one_year_weekly_return=log_weekly_return.iloc[log_weekly_return.index >= one_year_ago_date]
  weekly_std= np.std(one_year_weekly_return)*np.sqrt(one_year_weekly_return.shape[0])

  #Calculate garch_std sd for each stock
  one_year_monthly_return=log_monthly_return.iloc[log_monthly_return.index >= one_year_ago_date]
  res_all = one_year_monthly_return - np.mean(one_year_monthly_return)
  judge = one_year_monthly_return.shape[0]
  for stock in one_year_monthly_return.columns:
      res=res_all[stock]
      h= het_arch(res)
      if judge < 100 or h ==0:
          garch_std = weekly_std

      else:#build Garch Model
          std = []
          if judge <=200:
              p = 0
              q = 6
          else:
              p = 1
              q = 1

          std.append(fit_garch(one_year_monthly_return[stock],p,q))
          garch_std = pd.DataFrame(std,index = one_year_monthly_return.columns)

  #Calculate monthly sd for each stock
  one_year_monthly_return=log_monthly_return.iloc[log_monthly_return.index >= one_year_ago_date]
  monthly_std = np.std(one_year_monthly_return)*np.sqrt(12);

  #Calculate semi_monthly sd for each stock
  one_year_semi_monthly_return=log_semi_monthly_return.iloc[log_semi_monthly_return.index >= one_year_ago_date]
  semi_monthly_std = np.std(one_year_semi_monthly_return)*np.sqrt(24);

  # Calculated MaxDrop
  DD_range = 90
  total_range = 90
  DD_end = yahooDataFrame.iloc[yahooDataFrame.index <= one_year_ago_date]
  DD_start = yahooDataFrame.iloc[yahooDataFrame.index <= end-datetime.timedelta(total_range)]
  max_DD=[-1]*selected_stock_number


  if (DD_start.shape[0] != 1 ) and (DD_end.shape[0] != 1):
      DD_start = DD_start.index[-1]
      DD_end = DD_end.index[-1]
      while DD_start > DD_end :
          DD_start = yahooDataFrame.iloc[yahooDataFrame.index <= end-datetime.timedelta(total_range)]
          DD_mid = yahooDataFrame.iloc[yahooDataFrame.index <= end-datetime.timedelta(DD_range+total_range)]
          if (DD_start.shape[0] != 1 ) and (DD_mid.shape[0] != 1):
              DD_start = DD_start.index[-1]
              DD_mid = DD_mid.index[-1]
              price = yahooDataFrame.loc[DD_start:DD_mid,:]
              for i in range(len(yahooDataFrame.columns)):
                  max_DD[i]=max(max_DD[i],max_drawdown_absolute(stock_return_one_year.iloc[:,i])[0])
              total_range = total_range + 1
          else:
              break

  # Read OPT-data
  optdata = pd.read_excel("optdata_SP100_20220103_a.xlsx",index_col=0)
  P_current = MA_now
  P_fwd = optdata.P_fwd.loc[P_current.index]
  R_fwd=P_fwd/P_current-1
  R_60 = P_current/MA_one_year_ago - 1
  R_60_half = P_current/MA_half_year_ago - 1
  R_MA_1ago = R_mv_one_year
  R_MA_halfago = R_mv_half_year
  Rfwd_adjusted = pd.Series(index=R_MA_1ago.index)
  for i in R_fwd.index:
      if R_fwd[i] < 0.5*R_60[i]:
          Rfwd_adjusted[i] = R_60_half[i]
      elif R_fwd[i] > 1.3*R_60[i]:
          if 1.3*R_60[i] > 0.7*R_fwd[i]:
              Rfwd_adjusted[i] = 1.3*R_60[i]
          else:
              Rfwd_adjusted[i] = 0.7*R_fwd[i]
      else:
          Rfwd_adjusted[i] = R_fwd[i]
  Rfwd_adjusted[Rfwd_adjusted<0]=0

  price_now = today_price.iloc[-1,:]
  optimizer = optdata.loc[P_current.index,:]
  FFC = pd.DataFrame([R_MA_1ago.values,R_MA_halfago.values,MA_now.values,price_now.values,semi_monthly_std.values,weekly_std.values,MA_one_year_ago.values,MA_half_year_ago.values,max_DD,R_60.values,R_60_half.values,P_fwd.values,Rfwd_adjusted.values],columns = R_MA_1ago.index,index = ["R_MA_1ago","R_MA_halfago","MA_now","price",'std','std_f','MA_1ago','MA_halfago','Fall_MDD','R_60','R_60_half','R_fwd',"Rfwd_adjusted"]).T
  preselected_stock = preselected_stock.loc[selected_stock_list,:]
  preselected_stock["Rfwd_adjusted"]=Rfwd_adjusted.values
  preselected_stock["std_f"]=weekly_std.values
  preselected_stock["SR"]=preselected_stock["Rfwd_adjusted"]/preselected_stock["std_f"]
  preselected_stock["PR_SR"]=preselected_stock["PR"]*preselected_stock["SR"]**0.5
  preselected_stock.to_csv("preselected_Stocks_info.csv")
  p=FFC.MA_now
  EPS_fwd=optimizer.EPS_fwd
  EPS_ttm=optimizer.EPS_ttm
  fall=FFC.Fall_MDD
  FFC.to_csv("WEDataBank.csv")

  PE=p/EPS_fwd
  PEG= p/(EPS_fwd*(EPS_fwd/EPS_ttm-1))
  PEG[PEG<0]=2000

  cov = pd.DataFrame(np.cov(np.exp(one_year_weekly_return)-1,rowvar=False,bias = False)*51,index= one_year_weekly_return.columns, columns= one_year_weekly_return.columns)
  E_2 = Rfwd_adjusted/fall
  E_3 = Rfwd_adjusted/PEG
  E_4 = Rfwd_adjusted
  std =  np.std(np.exp(one_year_weekly_return)-1)*np.sqrt(52)

  """# Optimization"""

  w0 = 1/selected_stock_number*np.ones(selected_stock_number)
  uplim_ini = np.zeros(selected_stock_number)
  uplim3 =pd.DataFrame( w1*(1-uplim_ini) + w2 * uplim_ini,index = std.index,columns=["weight"])
  for i in uplim3.index:
      if i in ["SPY","IWD"]:
          uplim3.loc[i,:]=0
  low_lim =  np.zeros(selected_stock_number)
  multi=lambda x: x.T @cov@x

  ef = EfficientFrontier(E_2/100, cov,weight_bounds=(0, w1))
  #ef.add_constraint(lambda w: w@E_2== 12)
  raw_weights = ef.max_sharpe(0.01)
  cleaned_weights = ef.clean_weights()
  print(cleaned_weights)
  ef.portfolio_performance(verbose=True)

  weight1=np.array(pd.DataFrame(cleaned_weights.values(), index = list(cleaned_weights),columns = ["weight"]))[:,0]

  sd1 = np.sqrt(weight1@cov@weight1.T)
  R1 = weight1.T@E_2/100
  sharpe_ratio1 = (R1-0.01)/sd1
  print("new_sharpe:",sharpe_ratio1)

  """weight2 =   pd.read_excel("old_weight.xlsx",index_col=0)
  weight2=np.array(weight2.loc[list(cleaned_weights),:])[:,0]

  sd2 = np.sqrt(weight2@cov@weight2.T)
  R2 = weight2.T@E_2/100
  sharpe_ratio2 = (R2-0.01)/sd2
  print("old_sharpe:",sharpe_ratio2)
  """

  weight=pd.DataFrame(cleaned_weights.values(), index = list(cleaned_weights),columns = ["weight"])
  weight = weight[weight.weight >=0.030]
  weight.weight = weight.weight/sum(weight.weight)
  weight.to_csv("weights.csv")

  """See pyportfolioopt.readthedocs.io/en/latest/MeanVariance.html for more reference

  """

  if stock_type == "S&P100":
      OEX=pdr.get_data_yahoo("^OEX", start=start_swan_date, end=end_swan_date)["Adj Close"]

  elif stock_type == "S&P500":
      OEX=pdr.get_data_yahoo("SPY", start=start_swan_date, end=end_swan_date)["Adj Close"]

  stock_price=pdr.get_data_yahoo(list(weight.index), start=start_swan_date, end=end_swan_date)["Adj Close"]

  stock_price.to_csv("tradeinfo.csv")
  stock_price=stock_price[weight.index]
  stock_price

  weight

  start=stock_price.loc["2022-01-03"]
  end = stock_price.loc["2022-10-12"]
  stock_drop=abs(end/start-1)
  market_drop = abs(OEX.loc["2022-10-12"]/OEX.loc["2022-01-03"]-1)

  beta = stock_drop/market_drop
  1-beta.T@weight

  beta

  stock_drop

  rest=1
  extra=0
  for i, row in weight.iterrows():
    if row['weight']>0.17:
      rest=rest-row['weight']
      extra=extra+(row['weight']-0.17)
      # row['weight']=0.17

  for i,row in weight.iterrows():
    if row['weight']<0.165:
      # row['weight']=row['weight']+row['weight']/rest*extra
      a=1


  weights_name = f"Aweights_w1_{w1}_Seeds_{Seeds}.csv"
  weight

  portfolio_beta = portfolio_beta.loc[weight.index]

  swan_beta_name = f"A_forcast_and_actual_Beta{w1}_Seeds_{Seeds}.csv"

  portfolio_beta

  a=stock_price@weight
  b=1-a.loc["2022-10-12"]/a.loc["2022-01-03"]
  1-b/market_drop

  injected_result = pd.DataFrame(index = stock_price.index)
  S1 = np.array(10000*weight.T)/np.array(stock_price.iloc[0,:]).T

  portfolio_value = np.sum(np.array(weight.T) * stock_price,axis=1)
  S1 = np.array(10000*weight.T)/np.array(stock_price.iloc[0,:]).T
  injected_result["portfolio_value"]=stock_price@S1.T
  S2 = 10000/float(OEX[0])
  injected_result["oex"]=S2*OEX

  temp=injected_result.loc[["2022-01-03","2022-10-12"]]
  Portfolio_DD=abs(temp['portfolio_value'][1]/temp['portfolio_value'][0]-1)
  DD=abs(temp['oex'][1]/temp['oex'][0]-1)
  PR=1-(Portfolio_DD/DD)
  print(PR)
  temp

  result.loc[selected_stock_number,max_weight]=PR*100
  print(selected_stock_number,max_weight)

# Set up tests

In [10]:
n_preselected_stocks = np.arange(20,31,1)
max_weights = np.arange(0.05,0.16,0.01)

# Create a DataFrame with specified index and columns, initializing with zeros
result = pd.DataFrame(0, index=n_preselected_stocks, columns=max_weights)
for n_preselected_stock in n_preselected_stocks:
  for max_weight in max_weights:
    get_portfolio_PR(result,n_preselected_stock=n_preselected_stock,max_weight=max_weight)
# Display the DataFrame

['COP' 'XOM' 'CVX' 'LLY' 'MRK' 'BMY' 'LMT' 'AMGN' 'GD' 'ABBV' 'MET' 'KHC'
 'UNH' 'PEP' 'MO' 'RTX' 'JNJ' 'KO' 'GILD' 'PM']


[*********************100%%**********************]  20 of 20 completed
/usr/local/lib/python3.11/dist-packages/numpy/core/fromnumeric.py:3643: FutureWarning: The behavior of DataFrame.std with axis=None is deprecated, in a future version this will reduce over both axes and return a scalar. To retain the old behavior, pass axis=0 (or do not pass axis)
  return std(axis=axis, dtype=dtype, out=out, ddof=ddof, **kwargs)
/usr/local/lib/python3.11/dist-packages/numpy/core/fromnumeric.py:3643: FutureWarning: The behavior of DataFrame.std with axis=None is deprecated, in a future version this will reduce over both axes and return a scalar. To retain the old behavior, pass axis=0 (or do not pass axis)
  return std(axis=axis, dtype=dtype, out=out, ddof=ddof, **kwargs)
/usr/local/lib/python3.11/dist-packages/pypfopt/efficient_frontier/efficient_frontier.py:441: UserWarning: The risk_free_rate provided to portfolio_performance is different to the one used by max_sharpe. Using the previous value.
 

OrderedDict([('COP', 0.05), ('XOM', 0.05), ('CVX', 0.05), ('LLY', 0.05), ('MRK', 0.05), ('BMY', 0.05), ('LMT', 0.05), ('AMGN', 0.05), ('GD', 0.05), ('ABBV', 0.05), ('MET', 0.05), ('KHC', 0.05), ('UNH', 0.05), ('PEP', 0.05), ('MO', 0.05), ('RTX', 0.05), ('JNJ', 0.05), ('KO', 0.05), ('GILD', 0.05), ('PM', 0.05)])
Expected annual return: 1.4%
Annual volatility: 11.3%
Sharpe Ratio: 0.04
new_sharpe: 0.03787318975681855



[*********************100%%**********************]  20 of 20 completed
<ipython-input-9-0a1e38823244>:427: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  S2 = 10000/float(OEX[0])
<ipython-input-9-0a1e38823244>:431: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  Portfolio_DD=abs(temp['portfolio_value'][1]/temp['portfolio_value'][0]-1)
<ipython-input-9-0a1e38823244>:432: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  DD=abs(temp['oex'][

0.5680739051852541
20 0.05


...

[*********************100%%**********************]  30 of 30 completed
/usr/local/lib/python3.11/dist-packages/numpy/core/fromnumeric.py:3643: FutureWarning: The behavior of DataFrame.std with axis=None is deprecated, in a future version this will reduce over both axes and return a scalar. To retain the old behavior, pass axis=0 (or do not pass axis)
  return std(axis=axis, dtype=dtype, out=out, ddof=ddof, **kwargs)
/usr/local/lib/python3.11/dist-packages/numpy/core/fromnumeric.py:3643: FutureWarning: The behavior of DataFrame.std with axis=None is deprecated, in a future version this will reduce over both axes and return a scalar. To retain the old behavior, pass axis=0 (or do not pass axis)
  return std(axis=axis, dtype=dtype, out=out, ddof=ddof, **kwargs)
/usr/local/lib/python3.11/dist-packages/pypfopt/efficient_frontier/efficient_frontier.py:441: UserWarning: The risk_free_rate provided to portfolio_performance is different to the one used by max_sharpe. Using the previous value.
 

OrderedDict([('COP', 0.0), ('XOM', 0.0), ('CVX', 0.0), ('LLY', 0.0), ('MRK', 0.03902), ('BMY', 0.07408), ('LMT', 0.0), ('AMGN', 0.0), ('GD', 0.12652), ('ABBV', 0.02713), ('MET', 0.0), ('KHC', 0.0), ('UNH', 0.0), ('PEP', 0.0), ('MO', 0.0), ('RTX', 0.0), ('JNJ', 0.00957), ('KO', 0.0), ('GILD', 0.09265), ('PM', 0.0), ('SO', 0.0), ('WMT', 0.15), ('EXC', 0.15), ('MCD', 0.15), ('IBM', 0.0), ('CAT', 0.0), ('BRK-B', 0.15), ('MDLZ', 0.03103), ('AIG', 0.0), ('EMR', 0.0)])
Expected annual return: 4.5%
Annual volatility: 10.2%
Sharpe Ratio: 0.34
new_sharpe: 0.34279526166358415


[*********************100%%**********************]  9 of 9 completed


0.8542645547642895
30 0.15000000000000002


<ipython-input-9-0a1e38823244>:427: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  S2 = 10000/float(OEX[0])
<ipython-input-9-0a1e38823244>:431: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  Portfolio_DD=abs(temp['portfolio_value'][1]/temp['portfolio_value'][0]-1)
<ipython-input-9-0a1e38823244>:432: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  DD=abs(temp['oex'][1]/temp['oex'][0]-1)


In [11]:
result.to_csv("result.csv")
result

,0.05,0.06,0.07,0.08,0.09,0.10,0.11,0.12,0.13,0.14,0.15
20,56.807391,69.591018,76.179925,81.490905,95.822830,95.464580,96.676259,97.604231,98.557290,97.979360,97.449408
21,69.104730,76.251868,79.975017,98.213285,96.830643,98.712382,99.891665,99.882956,99.656966,99.418117,99.140417
22,72.707117,89.669388,97.071428,98.476583,99.533871,99.819274,98.912435,98.237772,97.726040,99.606707,99.997809
23,89.499703,99.160252,99.720896,98.656282,96.051890,95.087052,93.827251,93.819693,95.332945,95.283406,95.524290
24,95.539034,98.285489,97.545321,96.072859,92.679738,92.780925,92.171938,92.208300,91.952871,92.032574,91.994959
25,96.748731,96.466680,95.800482,94.097765,92.814534,90.737592,92.553789,92.002567,91.861082,91.806925,91.840920
26,96.748739,92.129870,95.801095,94.096880,92.814545,90.736689,92.553795,92.002580,91.861078,91.807252,91.840916
27,94.273634,0.000000,94.448771,92.506218,90.820212,90.091504,89.039894,88.687323,88.545841,89.020797,86.404467
28,92.022455,92.449520,91.003570,89.356445,87.141263,85.485874,85.429319,86.029825,86.928059,87.140993,85.426465
29,92.022456,92.449516,90.881759,89.356469,87.141218,85.485865,85.429283,86.029816,86.928059,87.141010,85.426460
